In [0]:
%pip install spark-nlp==5.2.3
%pip install langdetect

In [0]:
%restart_python

In [0]:
raw_review_df = (
    spark.read
         .option("multiLine", "false")
         .json("/Volumes/workspace/default/fashion_review/Fashion_reviews.jsonl")
)




In [0]:
raw_review_df.count()

2402363

In [0]:
raw_review_df.printSchema()

root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: long (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- year: long (nullable = true)



In [0]:
raw_review_df.show(3, truncate=False)

+----------+------------+-----------+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------+----------------------------+-----------------+----+
|asin      |helpful_vote|parent_asin|rating|text                                                                                                                                                                                                                                                                                                 |timestamp    |title        |user_id                     |verified_purchase|year|
+----------+------------+-----------+------+----------------------------------------------------------------------------------------------------------------------

In [0]:
display(raw_review_df)

Basic data cleaning (non-text)
(A) Drop duplicates

In [0]:
review_df = raw_review_df.dropDuplicates(["user_id", "asin", "title", "text"])


In [0]:
display(review_df)

In [0]:
review_df.count()

2378159

(B) Remove null or empty reviews

In [0]:
from pyspark.sql.functions import col, length, trim

df = review_df.filter(
    col("text").isNotNull() &
    (length(trim(col("text"))) > 0)
)


In [0]:
df.count()

2375767

In [0]:
display(df)

Text cleaning for sentiment analysis
(A) Combine title + text

In [0]:
from pyspark.sql.functions import concat_ws

df = df.withColumn(
    "review_text",
    concat_ws(" ", col("title"), col("text"))
)


(B) Normalize text
- Lowercase
- Remove HTML
- Remove punctuation
- Normalize unicode
- Remove extra spaces

In [0]:
from pyspark.sql.functions import lower, regexp_replace

df = (
    df.withColumn("review_text", lower(col("review_text")))
      .withColumn("review_text", regexp_replace("review_text", "<.*?>", ""))
      .withColumn("review_text", regexp_replace("review_text", "[^a-zA-Z\\s]", ""))
      .withColumn("review_text", regexp_replace("review_text", "\\s+", " "))
)


In [0]:
display(df)

Check language
(B)Remove Stopword

In [0]:
import sparknlp


In [0]:
df.select("review_text").show(3, truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
from langdetect import detect, LangDetectException
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType

@pandas_udf(StringType())
def detect_language_udf(texts: pd.Series) -> pd.Series:
    def safe_detect(text):
        try:
            return detect(text) if text and isinstance(text, str) else None
        except LangDetectException:
            return None
    return texts.apply(safe_detect)

In [0]:
df_lang = df.withColumn("detected_language", detect_language_udf(df["review_text"]))
df_lang.select("review_text", "detected_language").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
display(df_lang["review_text","detected_language"])

In [0]:
df_lang.groupBy("detected_language").count().orderBy("count", ascending=False).show()

+-----------------+-------+
|detected_language|  count|
+-----------------+-------+
|               en|2246382|
|               es|  26691|
|               ro|  19214|
|               af|   8850|
|               sv|   7931|
|               no|   7433|
|               ca|   7215|
|               fr|   7147|
|               da|   7035|
|               so|   5325|
|               it|   5155|
|               sl|   4507|
|               et|   2679|
|               sq|   2636|
|               pl|   2290|
|               nl|   2132|
|               pt|   2016|
|               cy|   1672|
|               cs|   1596|
|               hr|   1578|
+-----------------+-------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import col
en_df = df_lang.filter(col("detected_language") == "en")
en_df.count()

2246445

In [0]:
# Register df_lang as a temp view if not already done
df_lang.createOrReplaceTempView("df_lang")

# Translate non-English reviews to English using ai_translate (2 arguments only)
translated_df = spark.sql("""
SELECT
  asin,helpful_vote,parent_asin,rating,text,timestamp,title,user_id,verified_purchase,year,  
  review_text,
  detected_language,
  ai_translate(review_text, 'en') AS translated_review_text
FROM df_lang
WHERE detected_language != 'en'
""")

# Show sample results
translated_df.show(5, truncate=False)

+----------+------------+-----------+------+---------------------------------------------------------+-------------+------------------------------+----------------------------+-----------------+----+-------------------------------------------------------------+-----------------+----------------------------------------------------+
|asin      |helpful_vote|parent_asin|rating|text                                                     |timestamp    |title                         |user_id                     |verified_purchase|year|review_text                                                  |detected_language|translated_review_text                              |
+----------+------------+-----------+------+---------------------------------------------------------+-------------+------------------------------+----------------------------+-----------------+----+-------------------------------------------------------------+-----------------+----------------------------------------------------+
|

In [0]:
# Overwrite review_text with translated_review_text, then drop translated_review_text
translated_df = translated_df.withColumn("review_text", translated_df["translated_review_text"])
translated_df = translated_df.drop("translated_review_text")

In [0]:
#translated_df.count()

128028

In [0]:
#display(translated_df)

In [0]:
merged_df = en_df.unionByName(translated_df)


In [0]:
merged_df.count()


In [0]:
import re
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Define a set of English stopwords (can be expanded)
stopwords = set([
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours', 'yourself', 'yourselves',
    'he', 'him', 'his', 'himself', 'she', 'her', 'hers', 'herself', 'it', 'its', 'itself', 'they', 'them', 'their',
    'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those', 'am', 'is', 'are',
    'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an',
    'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about',
    'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up',
    'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when',
    'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no',
    'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 'can', 'will', 'just', 'don', 'should', 'now'
])

def remove_stopwords_regex(text):
    if not text:
        return ''
    # Split text into words using regex (words only)
    words = re.findall(r'\b\w+\b', text.lower())
    filtered = [word for word in words if word not in stopwords]
    return ' '.join(filtered)

remove_stopwords_udf = udf(remove_stopwords_regex, StringType())

# Apply to DataFrame
df_cleaned = merged_df.withColumn("cleaned_review_text", remove_stopwords_udf(merged_df["review_text"]))

In [0]:
display(df_cleaned.select("cleaned_review_text").limit(20))
